# 2. 工具的定义方式1：不使用 @tool

第 1 节的例子里，工具既有用 `@tool` 定义的，也有普通函数。本节专门研究后一种：**不加任何装饰器，普通 Python 函数也能直接交给 `bind_tools`。**

围绕这个现象，本节讲清楚四件事：

1. `bind_tools` 的每个参数是什么意思，它内部做了什么（2.2）
2. `convert_to_openai_tool` 如何把 Python 函数"翻译"成模型能看懂的工具说明（2.3）
3. 为什么不加 `@tool` 也能当工具，和加了 `@tool` 有什么区别（2.4）
4. 函数名、docstring、类型注解、默认值分别决定了工具说明的哪一部分（2.5）

## 2.1 模型绑定工具并发送请求

第一步：创建模型，定义一个普通函数作为工具。

In [1]:
import asyncio
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
from rich import print as rich_print
from langchain_openai import ChatOpenAI

# 从.env文件中加载环境变量
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]  # 与本项目 .env 中的名称一致



model= ChatOpenAI(
    model="gpt-6-luna",  # 当前配置已验证可用；原 gpt-5.5 返回模型权限 403
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)

from rich import print as rprint

# 定义工具
def get_weather(location: str):
    """获取指定位置的天气"""
    return f"{location}的天气是晴朗的"

第二步：绑定工具并提问。

In [1]:
model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke("今天北京的天气如何")
rprint(response)

AIMessage(
    content=[
        {
            'arguments': '{"location":"北京"}',
            'call_id': 'call_DEs0Ke4KULfvSLDCwthRaXOL',
            'name': 'get_weather',
            'type': 'function_call',
            'id': 'fc_019b1a80366ea0ea016ab685beb9b887d29851617fc92e2e56',
            'status': 'completed',
            'internal_chat_message_metadata_passthrough': {
                'create_time': 1790346686.159948,
                'turn_id': '01a0d8fa-699e-79d9-ae14-988160f59a75'
            },
            'metadata': {'turn_id': '01a0d8fa-699e-79d9-ae14-988160f59a75'}
        }
    ],
    additional_kwargs={},
    response_metadata={
        'id': 'resp_019b1a80366ea0ea016ab685bdb57887d2aea029f2cd0f46bc',
        'created_at': 1790346685.0,
        'metadata': {},
        'model': 'gpt-6-luna',
        'object': 'response',
        'service_tier': 'default',
        'status': 'completed',
        'model_provider': 'openai',
        'model_name': 'gpt-6-luna'
    },
    id='resp_019b1a80366ea0ea016ab685bdb57887d2aea029f2cd0f46bc',
    tool_calls=[
        {
            'name': 'get_weather',
            'args': {'location': '北京'},
            'id': 'call_DEs0Ke4KULfvSLDCwthRaXOL',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 4429,
        'output_tokens': 18,
        'total_tokens': 4447,
        'input_token_details': {'cache_creation': 0, 'cache_read': 0},
        'output_token_details': {'reasoning': 0}
    }
)

### 读懂上面的输出

输出很长，真正要看的是 `tool_calls` 字段：

```python
tool_calls=[{'name': 'get_weather', 'args': {'location': '北京'}, 'id': 'call_DEs0...', 'type': 'tool_call'}]
```

| 字段 | 含义 |
|---|---|
| `name` | 模型想调用哪个工具 |
| `args` | 模型根据问题填好的参数，已经解析成 dict |
| `id` | 这次调用的编号。执行完工具后，把结果放进 `ToolMessage(tool_call_id=...)` 交回模型时要用到 |

两点说明：

1. **模型没有执行 `get_weather`。** 它只是返回"我要调用 `get_weather`，参数是 `{"location": "北京"}`"，真正执行函数的是我们自己。因为这是普通函数，直接调用即可：`get_weather(**response.tool_calls[0]["args"])`。完整的"执行工具 → 回传结果"流程见 `01-Tools_usage.ipynb` 的 1.4 节。
2. **`content` 为什么是一个列表？** 这个中转服务走的是 OpenAI 的 **Responses API**，`content` 里放的是它的原始返回（`'type': 'function_call'`）。换成 DeepSeek 这类 Chat Completions 接口时，`content` 通常是空字符串（2.2.3 的例子可以看到）。无论哪种接口，LangChain 都会把工具调用统一整理到 `tool_calls` 里，所以**写代码时只读 `tool_calls`**。

## 2.2 bind_tools 详解

`bind_tools` 的作用：把工具"绑"到模型上。之后每次调用模型，都会自动把这些工具的说明一起发给模型。

### 2.2.1 看哪个签名？以实际运行的实现为准

在编辑器里查看 `bind_tools`，可能会看到不同版本的签名。它们来自继承链上的不同类：

| 类 | 角色 | 参数 |
|---|---|---|
| `BaseChatModel` | 所有聊天模型的父类，只定义接口，方法体只有一句 `raise NotImplementedError` | `tools`、`tool_choice` |
| `BaseChatOpenAI` | `ChatOpenAI` 的父类。**`ChatOpenAI` 实际运行的就是它的实现** | 多了 `strict`、`parallel_tool_calls`、`response_format` |
| `ChatDeepSeek` | 自己重写了一遍：`strict=True` 且使用官方地址时切换到 beta 接口，其余交给父类处理 | 签名里没有列出 `response_format` |

父类 `BaseChatModel` 的接口：

```python
def bind_tools(
    self,
    tools: Sequence[builtins.dict[str, Any] | type | Callable[..., Any] | BaseTool],
    *,
    tool_choice: str | None = None,
    **kwargs: Any,
) -> Runnable[LanguageModelInput, AIMessage]:
    raise NotImplementedError
```

`ChatOpenAI` 实际运行的实现（来自 `BaseChatOpenAI`）：

```python
def bind_tools(
    self,
    tools: Sequence[dict[str, Any] | type | Callable | BaseTool],
    *,
    tool_choice: dict | str | bool | None = None,
    strict: bool | None = None,
    parallel_tool_calls: bool | None = None,
    response_format: _DictOrPydanticClass | None = None,
    **kwargs: Any,
) -> Runnable[LanguageModelInput, AIMessage]
```

可以用代码确认实际用的是哪个类的实现：

In [2]:
import inspect
from langchain_deepseek import ChatDeepSeek

# __qualname__ 会显示方法定义在哪个类里
print(ChatOpenAI.bind_tools.__qualname__)
print(ChatDeepSeek.bind_tools.__qualname__)
print()
print(inspect.signature(ChatOpenAI.bind_tools))

BaseChatOpenAI.bind_tools
ChatDeepSeek.bind_tools

(self, tools: 'Sequence[dict[str, Any] | type | Callable | BaseTool]', *, tool_choice: 'dict | str | bool | None' = None, strict: 'bool | None' = None, parallel_tool_calls: 'bool | None' = None, response_format: '_DictOrPydanticClass | None' = None, **kwargs: 'Any') -> 'Runnable[LanguageModelInput, AIMessage]'


### 2.2.2 读懂 tools 参数的类型注解

```text
tools: Sequence[ builtins.dict[str, Any] | type | Callable[..., Any] | BaseTool ]
       ───┬────  ──────────┬──────────   ──┬─   ────────┬───────   ────┬───
          │                ①              ②            ③              ④
       一个序列，里面每个元素可以是 ①②③④ 中的任意一种
```

| 部分 | 含义 |
|---|---|
| `Sequence[...]` | 有顺序、能按下标取值的容器，比如 `list`、`tuple`。所以即使只有一个工具，也要写成 `[get_weather]` |
| `\|` | "或"。每个元素满足其中一种类型即可，同一个列表里可以混着放 |
| `Any` | 任意类型，不做限制 |

四种类型分别是：

**① `builtins.dict[str, Any]`：手写的工具说明字典**

键是字符串，值不限类型。`builtins` 是 Python 内置对象所在的模块，`builtins.dict` 就是平时用的 `dict`。之所以不直接写 `dict`，是因为 `BaseChatModel` 类里有一个同名的 `dict()` 方法，写成 `builtins.dict` 可以避免混淆。

**② `type`：一个类本身（不是类的实例）**

`type` 是"所有类的类型"，`isinstance(x, type)` 为 `True` 就说明 `x` 是一个类。实际能用的是 **Pydantic 类**和 **TypedDict 类**：类名变成工具名，docstring 变成描述，字段变成参数。

注意要传类本身 `GetWeather`，不能传实例 `GetWeather(location="北京")`。这种工具只有说明、没有函数体，常用于"只想让模型按固定结构输出数据"的场景。

**③ `Callable[..., Any]`：可调用对象，通常就是普通函数**

`Callable[参数, 返回值]` 表示能用 `()` 调用的对象。`...` 是 Python 的省略号，表示参数个数和类型不限。**本节的主角，不加装饰器的普通函数，就属于这一种。**

**④ `BaseTool`：LangChain 的工具对象**

用 `@tool` 装饰后得到的 `StructuredTool` 就是 `BaseTool` 的子类。

下面用五种写法定义同一个"查天气"工具，看看它们被转换后的结果：

In [3]:
from typing import TypedDict
from pydantic import BaseModel, Field
from langchain_core.tools import tool, BaseTool
from langchain_core.utils.function_calling import convert_to_openai_tool

# ① dict：手写好的工具说明
weather_dict = {
    "name": "get_weather",
    "description": "获取指定位置的天气",
    "parameters": {
        "type": "object",
        "properties": {"location": {"type": "string"}},
        "required": ["location"],
    },
}

# ② type：Pydantic 类
class GetWeather(BaseModel):
    """获取指定位置的天气"""
    location: str = Field(description="城市名")

# ② type：TypedDict 类
class GetWeatherTD(TypedDict):
    """获取指定位置的天气"""
    location: str

# ③ Callable：普通函数，就是 2.1 定义的 get_weather

# ④ BaseTool：@tool 装饰后的对象
@tool
def get_weather_tool(location: str) -> str:
    """获取指定位置的天气"""
    return f"{location}的天气是晴朗的"

for label, t in [
    ("① dict", weather_dict),
    ("② Pydantic 类", GetWeather),
    ("② TypedDict 类", GetWeatherTD),
    ("③ 普通函数", get_weather),
    ("④ @tool", get_weather_tool),
]:
    print(f"{label}：是类吗={isinstance(t, type)}，是 BaseTool 吗={isinstance(t, BaseTool)}")
    print("   ->", convert_to_openai_tool(t)["function"])

① dict：是类吗=False，是 BaseTool 吗=False
   -> {'name': 'get_weather', 'description': '获取指定位置的天气', 'parameters': {'type': 'object', 'properties': {'location': {'type': 'string'}}, 'required': ['location']}}
② Pydantic 类：是类吗=True，是 BaseTool 吗=False
   -> {'name': 'GetWeather', 'description': '获取指定位置的天气', 'parameters': {'properties': {'location': {'description': '城市名', 'type': 'string'}}, 'required': ['location'], 'type': 'object'}}
② TypedDict 类：是类吗=True，是 BaseTool 吗=False
   -> {'name': 'GetWeatherTD', 'description': '获取指定位置的天气', 'parameters': {'type': 'object', 'properties': {'location': {'type': 'string'}}, 'required': ['location']}}
③ 普通函数：是类吗=False，是 BaseTool 吗=False
   -> {'name': 'get_weather', 'description': '获取指定位置的天气', 'parameters': {'properties': {'location': {'type': 'string'}}, 'required': ['location'], 'type': 'object'}}
④ @tool：是类吗=False，是 BaseTool 吗=True
   -> {'name': 'get_weather_tool', 'description': '获取指定位置的天气', 'parameters': {'properties': {'location': {'type': 'string'}

In [4]:
# 传 Pydantic 实例而不是类，会报错
try:
    convert_to_openai_tool(GetWeather(location="北京"))
except ValueError as e:
    print("ValueError:", str(e).splitlines()[0])

ValueError: Unsupported function


五种写法得到的都是 `name + description + parameters` 结构。这就是注解里允许四种类型的原因：**LangChain 允许用多种方式描述工具，最后都会翻译成模型能看懂的同一种 JSON。**

### 2.2.3 其余参数

| 参数 | 含义 | 取值 |
|---|---|---|
| `*` | 分隔符。它后面的参数只能用关键字传，比如 `tool_choice="auto"`，不能按位置传 | — |
| `tool_choice` | 控制模型**是否调用、调用哪个**工具 | 见下表 |
| `strict` | 为 `True` 时，模型生成的参数**保证**符合 JSON Schema：不会多字段、少字段或类型错误。具体效果见 2.3.1 | `None`（默认，不发给 API）/ `True` / `False` |
| `parallel_tool_calls` | 为 `False` 时，模型一次最多调用一个工具。比如问"北京和上海天气"，默认可能一次返回两个 tool_call | `None`（默认，允许并行）/ `False` |
| `response_format` | 模型**没有**调用工具、直接回答时，让回答按指定结构输出。`ChatOpenAI` 专有 | Pydantic 类或 dict |
| `**kwargs` | 其他关键字参数原样交给 `bind()`，以后每次调用都会带上 | 很少用 |
| 返回值 `Runnable[LanguageModelInput, AIMessage]` | 一个可以 `.invoke()` 的对象：输入是字符串或消息列表（`LanguageModelInput`），输出是 `AIMessage` | — |

`tool_choice` 的取值：

| 写法 | 实际发给 API 的值 | 效果 |
|---|---|---|
| `None` / `False`（默认） | 不发送 | 使用 API 默认行为，通常等同 `"auto"` |
| `"auto"` | `"auto"` | 模型自己决定调不调、调哪个 |
| `"none"` | `"none"` | 禁止调用工具 |
| `"any"` / `"required"` / `True` | `"required"` | 至少调用一个工具，调哪个由模型决定 |
| `"get_weather"`（工具名） | `{"type": "function", "function": {"name": "get_weather"}}` | 必须调用这个工具 |

`"any"` 不是 OpenAI 原生支持的值，OpenAI 用的是 `"required"`。LangChain 会自动转换，所以不同厂商的模型都可以统一写 `"any"`。

下面的代码不会请求模型，只查看 `bind_tools` 最终记下了什么：

In [5]:
for choice in [None, "auto", "none", "any", True, "get_weather"]:
    bound = model.bind_tools([get_weather], tool_choice=choice)
    print(f"{choice!r:<15} -> {bound.kwargs.get('tool_choice', '（不发送）')}")

None            -> （不发送）
'auto'          -> auto
'none'          -> none
'any'           -> required
True            -> required
'get_weather'   -> {'type': 'function', 'function': {'name': 'get_weather'}}


实际效果：问一个和天气无关的问题"你好"，对比默认行为和强制调用。这里用 DeepSeek 演示（配置与 `01-Tools_usage.ipynb` 相同）：

In [6]:
ds_model = ChatDeepSeek(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    model_name="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}},
)

# 默认（相当于 auto）：模型判断不需要工具，直接回答
auto_resp = ds_model.bind_tools([get_weather]).invoke("你好")
print("默认        -> tool_calls:", auto_resp.tool_calls, "| content:", repr(auto_resp.content))

# 强制调用 get_weather：即使问题和天气无关，也必须返回工具调用
forced_resp = ds_model.bind_tools([get_weather], tool_choice="get_weather").invoke("你好")
print("get_weather -> tool_calls:", forced_resp.tool_calls, "| content:", repr(forced_resp.content))

默认        -> tool_calls: [] | content: '你好！有什么可以帮你的吗？比如查天气、聊聊天，我都可以。😊'


get_weather -> tool_calls: [{'name': 'get_weather', 'args': {'location': '北京'}, 'id': 'call_00_BtZz2PGzIhqp8ayTj9Mm4003', 'type': 'tool_call'}] | content: ''


对比两次结果：

- **默认**：问题和天气无关，模型直接回答，`tool_calls` 为空。
- **强制调用**：模型必须调用 `get_weather`，只好自己编了一个参数（`北京`）。此时 `content` 是空字符串，这就是 2.1 提到的 Chat Completions 接口的表现。

所以强制调用要慎用，它适合"这一步一定要用某个工具"的场景，比如借助工具做结构化信息抽取。

### 2.2.4 bind_tools 内部做了什么

去掉细节后，`BaseChatOpenAI.bind_tools` 的源码只有三步：

```python
# 1. 翻译：把每个工具转成 OpenAI 格式的 JSON 说明
formatted_tools = [convert_to_openai_tool(tool, strict=strict) for tool in tools]

# 2. 整理 tool_choice（就是 2.2.3 的那张转换表）
...

# 3. 绑定：把工具说明和其他参数"粘"到模型上
return super().bind(tools=formatted_tools, **kwargs)
```

可以记成一句话：**`bind_tools` = `convert_to_openai_tool`（翻译）+ `bind`（绑定）**。

第 3 步的 `bind()` **不会修改原来的 `model`**。它返回一个新的包装对象（`RunnableBinding`），里面记着"原模型"和"这组参数"。调用这个包装对象的 `.invoke()` 时，它把这组参数加进请求，再交给原模型。

好处是同一个 `model` 可以绑定不同的工具组合，得到多个互不影响的对象。下面的代码同样不请求模型：

In [7]:
from langchain_core.runnables import RunnableBinding

mwt = model.bind_tools([get_weather], parallel_tool_calls=False)

print(type(mwt).__name__, "是 RunnableBinding 吗:", isinstance(mwt, RunnableBinding))
print("包装的就是原来的 model:", mwt.bound is model)
print("绑定的参数:", list(mwt.kwargs))
rprint(mwt.kwargs["tools"])  # tools 已经是翻译好的 JSON

_ChatModelBinding 是 RunnableBinding 吗: True
包装的就是原来的 model: True
绑定的参数: ['tools', 'parallel_tool_calls']


[
    {
        'type': 'function',
        'function': {
            'name': 'get_weather',
            'description': '获取指定位置的天气',
            'parameters': {
                'properties': {'location': {'type': 'string'}},
                'required': ['location'],
                'type': 'object'
            }
        }
    }
]

> 补充：2.1 的输出说明这个中转服务走的是 Responses API。这种情况下，`ChatOpenAI` 在真正发请求前还会把上面的格式再"压平"一层：`{"type": "function", "function": {"name": ...}}` 变成 `{"type": "function", "name": ...}`。原理不变，知道有这回事即可。

## 2.3 convert_to_openai_tool：把工具翻译成 JSON 说明

模型看不懂 Python 函数，只认 JSON 格式的工具说明。`convert_to_openai_tool` 就是这个"翻译器"，`bind_tools` 的第 1 步就是调用它。官方文档的一句话介绍：Convert a tool-like object to an OpenAI tool schema（把类似工具的对象转换成 OpenAI 工具格式）。

### 2.3.1 函数签名与参数

```python
convert_to_openai_tool(
    tool: Mapping[str, Any] | type[BaseModel] | Callable[..., Any] | BaseTool,
    *,
    strict: bool | None = None,
) -> dict[str, Any]
```

| 参数 | 含义 |
|---|---|
| `tool` | 要翻译的**单个**工具。`bind_tools` 收的是列表，会对每个元素调用一次本函数。类型和 `bind_tools` 的 `tools` 元素基本一致：`Mapping` 是比 `dict` 更宽泛的"键值对"类型；`type[BaseModel]` 明确写出了"Pydantic 类"（TypedDict 类实际也支持） |
| `strict` | 严格模式。`None`（默认）：输出里不加 `strict` 字段；`True`：加上 `"strict": true`，并给 `parameters` 加上 `"additionalProperties": false`（不准出现未声明的参数）；`False`：只加上 `"strict": false` |
| 返回值 | 一个 dict，符合 OpenAI 工具格式 |

`bind_tools(..., strict=True)` 就是把 `strict` 原样传给了这个函数。先看普通模式：

In [8]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

def get_weather(city: str):
    """获取指定城市的天气"""
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))
rprint(convert_to_openai_tool(get_weather)['function']['parameters'])


{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取指定城市的天气',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

{'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}

再看严格模式，注意多出来的两个字段：

In [9]:
rprint(convert_to_openai_tool(get_weather, strict=True))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取指定城市的天气',
        'parameters': {
            'properties': {'city': {'type': 'string'}},
            'required': ['city'],
            'type': 'object',
            'additionalProperties': False
        },
        'strict': True
    }
}

### 2.3.2 输出结构：三个 type 分别是什么意思

输出里有三个 `type`，它们属于**两套不同的规则**：

In [ ]:
{
    'type': 'function',                 # ① OpenAI 工具格式：这是哪种工具
    'function': {
        'name': 'get_weather',
        'description': '获取指定城市的天气',
        'parameters': {                 # ↓ 从这里往里是 JSON Schema：参数长什么样
            'type': 'object',           # ② 所有参数合起来是一个 JSON 对象
            'properties': {
                'city': {'type': 'string'}   # ③ city 这个参数是字符串
            },
            'required': ['city']
        }
    }
}

**① `'type': 'function'`：工具的类别。** 这是 OpenAI 工具格式的字段，表示"这是开发者自己实现的函数工具"。OpenAI 还有 `web_search`、`file_search` 等内置工具，它们的 `type` 不是 `function`。

**② `'type': 'object'`：所有参数合起来是一个对象。** 从 `parameters` 往里用的是 **JSON Schema**，一种描述 JSON 数据结构的通用标准。`object` 是 JSON Schema 对"键值对集合"的叫法，相当于 Python 的 dict。它表示模型调用工具时，要把所有参数打包成 `{参数名: 值}`，2.1 输出里的 `'arguments': '{"location":"北京"}'` 就是这样一个对象。**`parameters` 最外层的 `type` 永远是 `object`。**

**③ `'type': 'string'`：单个参数的类型**，由类型注解 `city: str` 决定。

| 规则 | 管哪一层 | 回答的问题 |
|---|---|---|
| OpenAI 工具格式 | 最外层的 `type`、`function`、`name`、`description` | 这是什么工具、叫什么、做什么 |
| JSON Schema | `parameters` 里面 | 参数长什么样 |

常见 Python 类型会被转换成哪种 JSON Schema 类型？看一个例子：

In [10]:
def demo(city: str, days: int, temperature: float, detail: bool, tags: list[str], extra: dict):
    """演示各种类型注解的转换结果"""

rprint(convert_to_openai_tool(demo)["function"]["parameters"]["properties"])

{
    'city': {'type': 'string'},
    'days': {'type': 'integer'},
    'temperature': {'type': 'number'},
    'detail': {'type': 'boolean'},
    'tags': {'items': {'type': 'string'}, 'type': 'array'},
    'extra': {'additionalProperties': True, 'type': 'object'}
}

| Python 类型注解 | JSON Schema 类型 |
|---|---|
| `str` | `string` |
| `int` | `integer` |
| `float` | `number` |
| `bool` | `boolean` |
| `list[str]` | `array`，并用 `items` 说明元素类型 |
| `dict` | `object`。参数本身是 dict 时，`parameters` 里会再出现一个 `object` |

### 2.3.3 内部原理：按输入类型分发

`convert_to_openai_tool` 本身很短，大部分工作交给 `convert_to_openai_function`：

```text
convert_to_openai_tool(tool)
 ├─ 是 OpenAI 内置工具的 dict（如 {"type": "web_search"}）？ → 原样返回
 └─ 否则 → convert_to_openai_function(tool)，得到 {name, description, parameters}
          → 外面包一层 {"type": "function", "function": ...}
```

`convert_to_openai_function` 按顺序判断输入类型，走第一个匹配的分支：

| 顺序 | 输入 | 处理方式 |
|---|---|---|
| 1 | Anthropic 格式的 dict（有 `name` 和 `input_schema`） | 把 `input_schema` 改名为 `parameters` |
| 2 | Amazon Bedrock 格式的 dict（有 `toolSpec`） | 取出其中的 name、inputSchema |
| 3 | OpenAI 格式的 dict（有 `name`） | 只保留 `name`、`description`、`parameters`、`strict` |
| 4 | JSON Schema 格式的 dict（有 `title`） | `title` 作为工具名 |
| 5 | Pydantic 类 | 由 Pydantic 生成 JSON Schema |
| 6 | TypedDict 类 | 同上 |
| 7 | `BaseTool`（`@tool` 的产物） | 读取工具对象上已经准备好的参数模型 |
| 8 | 其他可调用对象（**普通函数**） | 见下文 |
| — | 都不匹配 | 抛出 `ValueError: Unsupported function` |

普通函数走第 8 个分支，由 `_convert_python_function_to_openai_function` 完成，分三步：

1. 取函数名 `get_weather`，作为工具名
2. 调用 `create_schema_from_function`：读取**函数签名**（参数名、类型注解、默认值）和 **docstring**（`parse_docstring=True`，解析 Google 风格的 `Args:`），**动态生成一个 Pydantic 模型**
3. 把这个 Pydantic 模型转成 JSON Schema，并去掉 Pydantic 自动加的 `title` 等字段

所以普通函数的转换，本质上是 **函数 → Pydantic 模型 → JSON Schema**。可以把中间那个 Pydantic 模型拿出来看看：

In [11]:
from langchain_core.tools.base import create_schema_from_function

def get_weather(city: str, unit: str = "摄氏度"):
    """获取指定城市的天气

    Args:
        city: 城市名称
        unit: 温度单位
    """
    return f"{city}天气晴朗"

# 第 2 步：函数 → Pydantic 模型
WeatherArgs = create_schema_from_function("get_weather", get_weather, parse_docstring=True)
print(WeatherArgs, "是 Pydantic 模型吗:", issubclass(WeatherArgs, BaseModel))
rprint(WeatherArgs.model_json_schema())

# 第 3 步之后：最终的工具说明，title 已经被去掉
rprint(convert_to_openai_tool(get_weather)["function"])

<class 'langchain_core.utils.pydantic.get_weather'> 是 Pydantic 模型吗: True


{
    'description': '获取指定城市的天气',
    'properties': {
        'city': {'description': '城市名称', 'title': 'City', 'type': 'string'},
        'unit': {'default': '摄氏度', 'description': '温度单位', 'title': 'Unit', 'type': 'string'}
    },
    'required': ['city'],
    'title': 'get_weather',
    'type': 'object'
}

{
    'name': 'get_weather',
    'description': '获取指定城市的天气',
    'parameters': {
        'properties': {
            'city': {'description': '城市名称', 'type': 'string'},
            'unit': {'default': '摄氏度', 'description': '温度单位', 'type': 'string'}
        },
        'required': ['city'],
        'type': 'object'
    }
}

## 2.4 为什么不加 @tool，普通函数也能当工具？

原因有三层。

**1. 模型只需要一份"说明书"，普通函数本身就带齐了。**

模型需要的只是 `name`、`description`、`parameters` 三样信息，普通函数里都有：

| 模型需要 | 普通函数里的来源 |
|---|---|
| `name` | 函数名 |
| `description` | docstring |
| `parameters` | 参数名 + 类型注解 + 默认值（+ docstring 里的 `Args:`） |

**2. LangChain 专门为普通函数准备了转换分支。**

`bind_tools` 的 `tools` 类型注解本来就包含 `Callable`。在 `convert_to_openai_function` 里，加了 `@tool` 的函数（已经变成 `BaseTool` 实例）走上面的分支；没加 `@tool` 的函数走下面的分支，现场根据函数定义和 docstring 生成 Pydantic 模型，再转换成规范的工具说明（2.3.3 表格的第 7、8 行）：

```python
elif isinstance(function, BaseTool):   # 加了 @tool：读取工具对象上的参数模型
    oai_function = cast("dict", _format_tool_to_openai_function(function))
elif callable(function):               # 普通函数：现场根据签名和 docstring 生成
    oai_function = cast("dict", _convert_python_function_to_openai_function(function))
```

两个分支殊途同归：`@tool` 在装饰时也是调用 `create_schema_from_function` 生成参数模型，只是提前生成好、存在了工具对象上。

**3. 模型从不执行函数。**

模型只返回 `tool_calls`（调用哪个工具、参数是什么），执行是我们自己的代码做的。所以对模型来说，函数有没有加 `@tool` 完全没有区别。

### 2.4.1 普通函数 vs @tool

既然都能用，`@tool` 多做了什么？

| 对比项 | 普通函数 | `@tool` |
|---|---|---|
| 对象类型 | `function` | `StructuredTool`（`BaseTool` 的子类） |
| 执行方式 | `get_weather("北京")` | `get_weather.invoke({"city": "北京"})`，不能直接用 `()` 调用 |
| docstring 里的 `Args:` | **默认解析**，写进每个参数的 `description` | **默认不解析**，整段 docstring 都塞进工具的 `description`；要写成 `@tool(parse_docstring=True)` 才解析 |
| 自定义工具名、描述、参数模型 | 不支持，只能改函数本身 | 支持，如 `@tool("工具名", description=..., args_schema=...)` |
| LangSmith 追踪 | 执行时不会单独记录 | `.invoke()` 会作为一次工具调用记录下来 |
| 交给 `create_agent(tools=[...])` | 支持，内部自动转成工具对象 | 支持 |

`@tool` 的写法下一节细讲。这里先用代码验证最容易踩坑的 docstring 解析差异：

In [12]:
def get_weather(city: str, unit: str = "摄氏度") -> str:
    """获取指定城市的天气

    Args:
        city: 城市名称
        unit: 温度单位
    """
    return f"{city}天气晴朗"

# tool(函数) 等价于在函数定义上面写 @tool
print("普通函数：")
rprint(convert_to_openai_tool(get_weather)["function"])
print("@tool（默认）：")
rprint(convert_to_openai_tool(tool(get_weather))["function"])
print("@tool(parse_docstring=True)：")
rprint(convert_to_openai_tool(tool(get_weather, parse_docstring=True))["function"])

普通函数：


{
    'name': 'get_weather',
    'description': '获取指定城市的天气',
    'parameters': {
        'properties': {
            'city': {'description': '城市名称', 'type': 'string'},
            'unit': {'default': '摄氏度', 'description': '温度单位', 'type': 'string'}
        },
        'required': ['city'],
        'type': 'object'
    }
}

@tool（默认）：


{
    'name': 'get_weather',
    'description': '获取指定城市的天气\n\n    Args:\n        city: 城市名称\n        unit: 温度单位',
    'parameters': {
        'properties': {'city': {'type': 'string'}, 'unit': {'default': '摄氏度', 'type': 'string'}},
        'required': ['city'],
        'type': 'object'
    }
}

@tool(parse_docstring=True)：


{
    'name': 'get_weather',
    'description': '获取指定城市的天气',
    'parameters': {
        'properties': {
            'city': {'description': '城市名称', 'type': 'string'},
            'unit': {'default': '摄氏度', 'description': '温度单位', 'type': 'string'}
        },
        'required': ['city'],
        'type': 'object'
    }
}

执行方式的区别：

In [13]:
weather_tool = tool(get_weather)

print(get_weather("北京"))                    # 普通函数：直接调用
print(weather_tool.invoke({"city": "北京"}))  # 工具对象：用 invoke，参数放在 dict 里

北京天气晴朗
北京天气晴朗


## 2.5 函数定义如何决定工具说明

普通函数的"说明书"来自函数名、docstring、类型注解和默认值。这一节逐项看它们对应输出的哪一部分，以及写错了会怎样。

### 2.5.1 函数名与 description

- **函数名 → `name`**。模型靠 `name` 和 `description` 判断该调用哪个工具，函数名要见名知意，比如 `get_weather` 比 `func1` 好得多。
- **docstring → `description`**。convert_to_openai_tool 会从 docstring(文档字符串) 加载工具的描述信息。2.3 的案例中，docstring 为"获取指定城市的天气"，所以抽取的 description 为"获取指定城市的天气"。

> docstring，文档字符串，使用三个双引号表示开始和结束，写在函数体的第一行。

更细的规则如下（下面的代码会验证）：

| docstring 的内容 | 会不会进入 description |
|---|---|
| `Args:` 之前的文字 | 会。多段文字会合并成一行 |
| `Args:` 部分 | 不会，它变成各个参数的 `description`（见 2.5.2） |
| `Returns:`、`Raises:` 部分 | **不会**，模型看不到。想让模型知道的信息，要写在 `Args:` 之前 |
| 没有 docstring | description 为空字符串，模型只能靠函数名猜工具的用途 |

In [14]:
def get_weather(city: str) -> str:
    """获取指定城市的天气。

    返回当天的天气状况和温度。

    Args:
        city: 城市名称

    Returns:
        天气信息字符串

    Raises:
        ValueError: 城市不存在时抛出
    """
    return f"{city}天气晴朗"

def get_weather_no_doc(city: str) -> str:
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather)["function"])
rprint(convert_to_openai_tool(get_weather_no_doc)["function"])

{
    'name': 'get_weather',
    'description': '获取指定城市的天气。 返回当天的天气状况和温度。',
    'parameters': {
        'properties': {'city': {'description': '城市名称', 'type': 'string'}},
        'required': ['city'],
        'type': 'object'
    }
}

{
    'name': 'get_weather_no_doc',
    'description': '',
    'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
}

### 2.5.2 参数说明：Google 风格 docstring

convert_to_openai_tool 会从 docstring 加载参数说明，这里的 docstring 必须遵循 **Google 风格**：

- 用 `Args:`、`Returns:`、`Raises:` 等关键字开头，各占一行
- `Args:` 下面每个参数占一行，缩进，格式为 `参数名: 说明`

下面是 Google 风格的标准示例：

In [15]:
def function_with_pep484_type_annotations(param1: int, param2: str) -> bool:
    """Example function with PEP 484 type annotations.

    Args:
        param1: The first parameter.
        param2: The second parameter.

    Returns:
        The return value. True for success, False otherwise.

    """
    pass #此为举例

按这个格式写 `get_weather`，参数 `city` 的说明就进入了 `parameters`：

In [16]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

def get_weather(city: str):
    """获取指定城市的天气
    
    Args:
        city: 城市名称。

    Returns:
        当前城市天气。
    
    """
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))
#print(convert_to_openai_tool(get_weather)['function']['parameters'])


{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取指定城市的天气',
        'parameters': {
            'properties': {'city': {'description': '城市名称。', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

使用 Args: 、 Returns: 、 Raises: 等关键字，这种方式可读性强。Agent通过工具的这些注释来理解工具的用途和调用时机，因此清晰、准确的文档字符串是工具能被正确调用的前提。

注意：发给模型的只有 `Args:` 之前的描述和 `Args:` 里的参数说明，`Returns:`、`Raises:` 主要是写给人看的（见 2.5.1）。

**如果不按 Google 风格写，会怎样？** 不会报错，但 `Args` 识别不出来：整段 docstring（包括参数说明）都被当成工具描述，参数本身没有 `description`。比如 `01-Tools_usage.ipynb` 里用过的中文写法 `参数:`：

In [17]:
def get_weather(city: str) -> str:
    """获取指定城市的天气
    参数:
        city: 城市名称
    """
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather)["function"])

{
    'name': 'get_weather',
    'description': '获取指定城市的天气\n参数:\n    city: 城市名称',
    'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
}

### 2.5.3 参数类型说明

参数类型来源于函数的**类型注解**（Python 类型和 JSON Schema 类型的对应关系见 2.3.2）。如果删除了参数类型注解，则工具描述中不包含参数类型说明。

没有类型注解时分两种情况：

| 情况 | 结果 |
|---|---|
| docstring 的 `Args:` 里**写了**这个参数 | **报错** `ValueError` |
| docstring 里**没写**这个参数 | 不报错，但参数的说明是空的 `{}`，模型不知道该传什么类型 |

注意：如果docstring中包含参数说明，则对应的参数必须有类型注解，否则报错。

In [18]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
def get_weather(city):
    """
    天气查询工具

    Args:
        city: 城市名称
    """
    print("天气晴朗")

try:
    rprint(convert_to_openai_tool(get_weather))
except ValueError as e:
    print("ValueError:", e)

ValueError: Arg city in docstring not found in function signature.


报错信息说"`city` 不在函数签名里"，可 `city` 明明就在签名里，这句话有误导性。看源码就明白了：LangChain 解析出 `Args:` 里的参数后，会逐个检查它**有没有类型注解**，没有注解就当成"找不到"：

```python
# langchain_core/tools/base.py
def _validate_docstring_args_against_annotations(arg_descriptions, annotations):
    for docstring_arg in arg_descriptions:      # docstring 里写了的参数
        if docstring_arg not in annotations:    # annotations 只包含有类型注解的参数
            msg = f"Arg {docstring_arg} in docstring not found in function signature."
            raise ValueError(msg)
```

所以看到这个报错，先检查对应参数有没有写类型注解。

反过来，没有类型注解、docstring 也不提这个参数时，不会报错，但参数说明是空的：

In [19]:
def get_weather(city):
    """天气查询工具"""
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather)["function"]["parameters"])

{'properties': {'city': {}}, 'required': ['city'], 'type': 'object'}

### 2.5.4 参数默认值说明

如果参数没有默认值，则会包含在 required 对应的列表中。

反之，则参数的描述信息会包含 default 字段，并且不会出现在 required 列表中。

In [20]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
def get_weather(city: str="北京"):
    """
    天气查询工具
    
    Args:
        city: 城市名称

    Returns:
        天气信息
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '天气查询工具',
        'parameters': {
            'properties': {'city': {'default': '北京', 'description': '城市名称', 'type': 'string'}},
            'type': 'object'
        }
    }
}

补充三点：

1. 上面的例子里唯一的参数有默认值，所以输出中**连 `required` 键都没有**。
2. `required` 告诉模型哪些参数必须填。不在 `required` 里的参数，模型可以不传，执行函数时就用默认值。`default` 字段则让模型知道"不传的话默认是什么"。
3. 参数允许为空时写成 `unit: str | None = None`，类型会变成 `anyOf`，意思是"字符串或 null 二选一"：

In [21]:
def get_weather(city: str, unit: str | None = None) -> str:
    """获取指定城市的天气

    Args:
        city: 城市名称
        unit: 温度单位，不传则使用摄氏度
    """
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather)["function"]["parameters"])

{
    'properties': {
        'city': {'description': '城市名称', 'type': 'string'},
        'unit': {
            'anyOf': [{'type': 'string'}, {'type': 'null'}],
            'default': None,
            'description': '温度单位，不传则使用摄氏度'
        }
    },
    'required': ['city'],
    'type': 'object'
}

## 2.6 总结

**一条主线：**

```text
def get_weather(city: str): """..."""          ← 普通函数，不加 @tool
        │  bind_tools 第 1 步：convert_to_openai_tool
        │  （函数 → Pydantic 模型 → JSON Schema）
        ▼
{"type": "function", "function": {name, description, parameters}}
        │  bind_tools 第 3 步：bind()，返回新的 RunnableBinding，原 model 不变
        ▼
model_with_tools.invoke("今天北京天气如何")      ← 工具说明随请求一起发给模型
        ▼
AIMessage.tool_calls = [{"name": "get_weather", "args": {"city": "北京"}, "id": ...}]
        │  模型只"开调用单"，不执行
        ▼
get_weather(**args)                           ← 由我们自己执行
```

**要点：**

1. `bind_tools` = `convert_to_openai_tool`（翻译）+ `bind`（绑定）。它不修改原模型，而是返回一个记住了工具参数的新对象。
2. 看签名要以实际运行的实现为准：`ChatOpenAI` 用的是 `BaseChatOpenAI.bind_tools`，比父类 `BaseChatModel` 多了 `strict`、`parallel_tool_calls`、`response_format`。
3. `tools` 列表里可以放 dict、类（Pydantic / TypedDict，传类本身而不是实例）、普通函数、`BaseTool`，最后都翻译成同一种 JSON。
4. `tool_choice` 控制调不调工具：`"auto"` 由模型决定，`"none"` 禁止调用，`"any"` / `True` 至少调一个，写工具名则必须调用它。
5. 工具说明最外层的 `type: function` 属于 OpenAI 工具格式；`parameters` 里面属于 JSON Schema，其最外层永远是 `type: object`。
6. 普通函数能当工具，是因为模型只需要说明书，而函数名、docstring、类型注解已经提供了全部信息；执行始终由我们负责。
7. 普通函数默认解析 Google 风格的 `Args:`；`@tool` 默认不解析，要加 `parse_docstring=True`。

**写一个不加 @tool 的工具，检查清单：**

- [ ] 函数名见名知意（它就是工具名）
- [ ] docstring 第一段写清"做什么、什么时候用"（`Returns:` 模型看不到）
- [ ] 用 Google 风格的 `Args:` 给每个参数写说明
- [ ] 每个参数都有类型注解（`Args:` 里写了、却没有注解的参数会报错）
- [ ] 可选参数给默认值（它们不会出现在 `required` 里）